In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df = df.fillna(df.mean())

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
print("Number of duplecated: ",df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include="object").columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

standard_scaler = StandardScaler() # Instantiate StandardScaler

featcher_cols = df.drop("Target",axis=1).columns

df[featcher_cols] = standard_scaler.fit_transform(df[featcher_cols])

In [ ]:
df.head()

In [ ]:
# Task 5: Write your code here:
df["Target"].hist()
# Answer: Targe is not impalance

In [ ]:
!pip install catboost

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target",axis=1)
y = df["Target"]


In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

results = {'accuracy': [], 'f1': []}

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training ...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  results['accuracy'].append(accuracy)
  results['f1'].append(f1)
  print(f"  Accuracy:  {np.mean(results['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:

importances = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
features = X.columns


  # Sort features by importance for a cleaner plot
sorted_idx = np.argsort(importances)

ax = axes[0]
ax.barh(features[sorted_idx], importances[sorted_idx])
ax.set_title(f"Feature Importance")
ax.set_xlabel("Importance Score")

plt.show()

In [ ]:
# Task 2: Write your code here:
features[sorted_idx][0]

In [ ]:
# Task Bonus: Write your code here:
X_train_new = pd.DataFrame(X["D_68"])

new_results = {'accuracy': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X_train_new.iloc[train_index], X_train_new.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  print(f"Training ...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  new_results['accuracy'].append(accuracy)
  new_results['f1'].append(f1)
  print(f"  Accuracy:  {np.mean(new_results['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(new_results['f1']):.4f}")

In [ ]:
print("model With All featchers:")
print(f"  Accuracy:  {np.mean(results['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(results['f1']):.4f}")
print("model with only goldest featcher:")
print(f"  Accuracy:  {np.mean(new_results['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(new_results['f1']):.4f}")

In [ ]:
# So, in this case i think almost accuercty get from gold feacther but other featcher requierd to improve accuercy <_>